# Multithreading

In [ ]:
!pip install pytubefix

In [ ]:

import threading
import os
import queue
from tqdm.auto import tqdm
from functools import partial
from pytubefix import YouTube
import time
from threading import Lock
import concurrent.futures


### Parallelism Concept

- Sequential vs Parallel

In [ ]:
def say_numbers():
    # Count 1~5
    for i in range(1,6,1):
        print(f"Number: #{i}")
        time.sleep(0.1)

def say_alphabet():
    # Count a~e
    for i in range(ord("a"), ord("f")):
        print(f"Alphabet: {chr(i)}")
        time.sleep(0.1)

In [ ]:

print("-"*10+"Sequential execution"+"-"*10)
seq = time.time()
say_numbers()
say_alphabet()
print("execution time :", time.time() - seq)


In [ ]:
print("-"*10+"Parallel execution"+"-"*10)
par = time.time()
t1 = threading.Thread(target=say_numbers)
t2 = threading.Thread(target=say_alphabet)
t1.start()
t2.start()
t1.join()
t2.join()
print("execution time :", time.time() - par)

- Accessing memory

In [ ]:
import multiprocessing

def append_one(l):
    l.append(1)

def append_two(l):
    l.append(2)

In [ ]:
# A list is sharable for multiple threads t1 and t2.
print("-"*10+"Multi-threading"+"-"*10)
list1 = []
t1 = threading.Thread(target=append_one, args=(list1,))
t2 = threading.Thread(target=append_two, args=(list1,))
t1.start()
t2.start()
t1.join()
t2.join()
print(f"Multi-threading result:{list1}")

In [ ]:
# Processes doesn't share data (generally)
print("-"*10+"Multi-processing"+"-"*10)
list2 = []
process1 = multiprocessing.Process(target=append_one, args=(list2,))
process2 = multiprocessing.Process(target=append_two, args=(list2,))
process1.start()
process2.start()
process1.join()
process2.join()
print(f"Multi-processing result:{list2}")

### Race Condition

In [ ]:
# Shared variable across threads.
balance = 1000


def withdraw_unsafe(amount):
    global balance

    if balance >= amount:
        time.sleep(0.001)  # GIL releases here
        balance = balance - amount  # Write stale value
        print(f"Withdrew {amount}, balance = {balance}")
    else:
        print(f"REJECTED (balance={balance})")


print("-" * 10 + "Race condition (no lock)" + "-" * 10)
print(f"Starting balance: {balance}")

threads = [
    threading.Thread(target=withdraw_unsafe, args=(700,)),
    threading.Thread(target=withdraw_unsafe, args=(700,)),
]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"Final balance: {balance}")

In [ ]:
# Shared variable across threads.
balance = 1000
lock = threading.Lock()

def withdraw_safe(amount):
    global balance, lock

    with lock:
        if balance >= amount:
            time.sleep(0.001)  # GIL releases here
            balance = balance - amount  # Write stale value
            print(f"Withdrew {amount}, balance = {balance}")
        else:
            print(f"REJECTED (balance={balance})")


print("-" * 10 + "Fixed" + "-" * 10)
print(f"Starting balance: {balance}")

threads = [
    threading.Thread(target=withdraw_safe, args=(700,)),
    threading.Thread(target=withdraw_safe, args=(700,)),
]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"Final balance: {balance}")

### `pytube` `tqdm` example - using variable typed Queue

- UI thread

In [ ]:
def draw_ui(var):
    print("UI thread starting ... PID:{}".format(os.getpid()))
    prev = 0
    tqdm_bar = None
    while True:
        message = var.get()
        if message["type"] == "on_progress":
            if tqdm_bar is None:
                tqdm_bar = tqdm(total=100, desc="Downloading...")
            cur_rate = message["progress_rate"]
            tqdm_bar.update(int(cur_rate-prev))
            prev = int(cur_rate)
        elif message["type"] == "on_complete":
            if tqdm_bar is None:
                tqdm_bar = tqdm(total=100, desc="Downloading...")
            tqdm_bar.update(100-prev)
            tqdm_bar.close()
            break

- Download thread

In [ ]:
def on_progress(stream, chunk, bytes_remaining, var):
    total_size = stream.filesize
    bytes_downloaded = total_size - bytes_remaining
    progress = (bytes_downloaded / total_size) * 100
    var.put({"type":"on_progress", "progress_rate":progress})

def on_complete(stream, file_handle, var):
    var.put({"type":"on_complete"})

def download(url, var):
    print("Download thread starting ... PID:{}".format(os.getpid()))
    on_progress_with_Q = partial(on_progress, var=var)
    on_complete_with_Q = partial(on_complete, var=var)
    youtube_clip = YouTube(
                        url,
                        on_progress_callback=on_progress_with_Q,
                        on_complete_callback=on_complete_with_Q)
    youtube_stream = youtube_clip.streams.get_highest_resolution()
    youtube_stream.download("videos")

- Multithreading

In [ ]:
# 코난 오브라이언
url = "https://youtu.be/9PPifK4d0TA?si=Mp3k-L1IyPf2tUl_"

print("main process running ... PID:{}".format(os.getpid()))

shared_var = queue.Queue()  # message_queue = multiprocessing.Queue()가 아니라 그냥 queue.Queue()로 사용

t1 = threading.Thread(target=draw_ui, args=(shared_var,))
t2 = threading.Thread(target=download, args=(url, shared_var,))

t1.start()
t2.start()

t1.join()
shared_var.put(None)
t2.join()